# Visual Novel — Render video bằng Google Colab

Quy trình rút gọn: **Kết nối runtime → Chạy tất cả (Run all) → Chọn ZIP khi được hỏi**. Colab tự cài thư viện, kết nối Drive, nhận gói ZIP từ `phong_video.html`, tải engine và render.

Kết quả gồm MP4, SRT, Audio Manifest, metadata và một ZIP tổng hợp; được lưu trong `MyDrive/VisualNovelRenders`. Gói ZIP đầu vào chỉ chứa timeline/WAV và tham chiếu tài nguyên; engine tải tài nguyên gốc từ GitHub.

**Lưu ý:** Colab miễn phí có giới hạn thời gian và tài nguyên, GPU không được đảm bảo. Hãy thử một đoạn ngắn trước; bản render native có thể chưa giống pixel-perfect với preview trình duyệt.

In [ ]:
import sys, subprocess, re, zipfile
from pathlib import Path

print('1/5 Cài thư viện render…')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'av', 'opencv-python-headless', 'pillow', 'numpy', 'requests'], check=True)

from google.colab import drive, files
import requests
print('2/5 Kết nối Google Drive…')
drive.mount('/content/drive')

print('3/5 Chọn gói ZIP từ phòng tạo video…')
uploaded = files.upload()
zip_names = [name for name in uploaded if name.lower().endswith('.zip')]
if not zip_names:
    raise RuntimeError('Chưa chọn file *_colab_render_package.zip.')
ZIP_PATH = Path('/content') / zip_names[0]
print('Đã nhận:', ZIP_PATH.name, round(ZIP_PATH.stat().st_size/1024/1024, 1), 'MB')

print('4/5 Tải engine render từ GitHub…')
ENGINE_URL = 'https://raw.githubusercontent.com/nhothicam01-png/visual-novel-game/main/colab-render/renderer.py'
response = requests.get(ENGINE_URL, timeout=90)
response.raise_for_status()
ENGINE_PATH = Path('/content/renderer.py')
ENGINE_PATH.write_text(response.text, encoding='utf-8')

OUTPUT_DIR = Path('/content/drive/MyDrive/VisualNovelRenders')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
base = re.sub(r'[^A-Za-z0-9_-]+', '_', ZIP_PATH.stem.replace('_colab_render_package','')) or 'VisualNovel'
OUTPUT_PATH = OUTPUT_DIR / (base + '_colab.mp4')
print('5/5 Bắt đầu render…')
subprocess.run([sys.executable, '-u', str(ENGINE_PATH), '--zip', str(ZIP_PATH), '--output', str(OUTPUT_PATH)], check=True)

sidecars = [OUTPUT_PATH.with_suffix('.srt'), OUTPUT_PATH.with_name(OUTPUT_PATH.stem+'_audio_manifest.json'), OUTPUT_PATH.with_name(OUTPUT_PATH.stem+'_metadata.json')]
BUNDLE_PATH = OUTPUT_DIR / (OUTPUT_PATH.stem + '_all_files.zip')
with zipfile.ZipFile(BUNDLE_PATH, 'w', zipfile.ZIP_DEFLATED) as bundle:
    for path in [OUTPUT_PATH] + sidecars:
        if path.exists(): bundle.write(path, arcname=path.name)
print('Hoàn tất! File MP4 và phụ trợ nằm tại:', OUTPUT_DIR)
print('Gói tải tất cả:', BUNDLE_PATH)

## Tải kết quả về điện thoại (tùy chọn)

Ô dưới đây tải một ZIP gồm MP4, SRT, Audio Manifest và metadata. Nếu video lớn, mở thư mục Google Drive `VisualNovelRenders` để tải sau.

In [ ]:
from google.colab import files
files.download(str(BUNDLE_PATH))